# DMV Practical 3 – Telco Customer Churn Data Preprocessing

**Aim:** Perform data preprocessing on the Telco Customer Churn dataset: inspect the data, identify missing and duplicate values, standardize categorical data, convert data types, handle missing values and outliers, create new features, visualize churn patterns, standardize numerical features, and separate features and target.

**Dataset:** `WA_Fn-UseC_-Telco-Customer-Churn.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully!")

In [ ]:
url = "https://raw.githubusercontent.com/Giskard-AI/examples/main/datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

In [ ]:
print("Shape of dataset:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

In [ ]:
df.info()

In [ ]:
display(df.describe(include="all"))

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

missing_percentage = (df.isnull().sum() / len(df)) * 100

print("Missing value percentage:")
print(missing_percentage)

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicates removed.")
print("New shape:", df.shape)

In [ ]:
categorical_columns = df.select_dtypes(include=["object"]).columns

for column in categorical_columns:
    print("\n", column)
    print(df[column].value_counts().head(20))

In [ ]:
for column in categorical_columns:
    df[column] = df[column].astype(str).str.strip()

print("Text formatting standardized.")

for column in categorical_columns:
    if set(df[column].dropna().unique()).issubset(
        {"Yes", "No", "yes", "no", "YES", "NO"}
    ):
        df[column] = df[column].str.title()

print("Text formatting standardized.")

In [ ]:
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"], errors="coerce"
    )

if "tenure" in df.columns:
    df["tenure"] = pd.to_numeric(
        df["tenure"], errors="coerce"
    )

print(df.dtypes)

In [ ]:
print(df.isnull().sum())

numeric_columns = df.select_dtypes(include=np.number).columns

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

categorical_columns = df.select_dtypes(include=["object"]).columns

for column in categorical_columns:
    if df[column].isnull().sum() > 0:
        df[column] = df[column].fillna(df[column].mode()[0])

print("Missing values handled.")
print(df.isnull().sum())

In [ ]:
print("Remaining missing values:")
print(df.isnull().sum().sum())

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

for column in numeric_columns:
    plt.figure(figsize=(5, 3))
    sns.boxplot(x=df[column])
    plt.title("Box Plot - " + column)
    plt.tight_layout()
    plt.show()

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[column] = df[column].clip(
        lower_bound,
        upper_bound
    )

print("Outliers handled using IQR capping.")

In [ ]:
if "tenure" in df.columns:
    df["TenureGroup"] = pd.cut(
        df["tenure"],
        bins=[-1, 12, 24, 48, 72, np.inf],
        labels=[
            "0-1 Year",
            "1-2 Years",
            "2-4 Years",
            "4-6 Years",
            "6+ Years"
        ]
    )

display(df.head())

In [ ]:
if "MonthlyCharges" in df.columns:
    df["EstimatedAnnualCharges"] = (
        df["MonthlyCharges"] * 12
    )

display(df.head())

In [ ]:
service_columns = [
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

existing_services = [
    col for col in service_columns
    if col in df.columns
]

for col in existing_services:
    df[col] = df[col].astype(str)

df["ServiceCount"] = 0

for col in existing_services:
    df["ServiceCount"] += (
        df[col].str.lower().isin(["yes", "fiber optic", "dsl"])
    ).astype(int)

display(df[["ServiceCount"]].head())

In [ ]:
if "Churn" in df.columns:
    print("Churn distribution:")
    print(df["Churn"].value_counts())

    print("\nChurn percentage:")
    print(df["Churn"].value_counts(normalize=True) * 100)

In [ ]:
if "Churn" in df.columns:
    plt.figure(figsize=(5, 3))

    df["Churn"].value_counts().plot(kind="bar")

    plt.title("Customer Churn Distribution")
    plt.xlabel("Churn")
    plt.ylabel("Number of Customers")
    plt.tight_layout()
    plt.show()

In [ ]:
if "Contract" in df.columns and "Churn" in df.columns:
    contract_churn = pd.crosstab(
        df["Contract"],
        df["Churn"],
        normalize="index"
    ) * 100

    display(contract_churn)

    contract_churn.plot(
        kind="bar",
        figsize=(5, 3)
    )

    plt.title("Churn Rate by Contract Type")
    plt.xlabel("Contract")
    plt.ylabel("Churn Percentage")
    plt.tight_layout()
    plt.show()

In [ ]:
if "PaymentMethod" in df.columns and "Churn" in df.columns:
    payment_churn = pd.crosstab(
        df["PaymentMethod"],
        df["Churn"],
        normalize="index"
    ) * 100

    display(payment_churn)

In [ ]:
if "MonthlyCharges" in df.columns and "Churn" in df.columns:
    plt.figure(figsize=(5, 3))

    sns.boxplot(
        x="Churn",
        y="MonthlyCharges",
        data=df
    )

    plt.title("Monthly Charges vs Churn")
    plt.xlabel("Churn")
    plt.ylabel("Monthly Charges")
    plt.tight_layout()
    plt.show()

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

scaler = StandardScaler()
scaled_data = df.copy()

scaled_data[numeric_columns] = scaler.fit_transform(
    scaled_data[numeric_columns]
)

print("Numerical features standardized.")
display(scaled_data.head())

In [ ]:
if "Churn" in df.columns:
    X = df.drop("Churn", axis=1)
    y = df["Churn"]

    print("Features shape:", X.shape)
    print("Target shape:", y.shape)

In [ ]:
if y.dtype == "object":
    y = y.map({
        "Yes": 1,
        "No": 0
    })

print("Target values encoded.")
print(y.value_counts())

In [ ]:
# Final dataset check
print("Final dataset shape:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())